# Self-Disclosure Detection Pipeline

Runs annotation, training, and the experiment grid on a Colab GPU.

**Runtime > Change runtime type > T4 GPU** before starting.

## Two modes

`DRY_RUN = True` uses an innocuous public text dataset to verify the whole
pipeline end to end and measure throughput. Run this first. It touches no
sensitive data and needs no ethics approval, so it can be done while the
Secondary Data Checklist is still with the supervisor.

`DRY_RUN = False` uses the real corpus. **Do not switch this until the
checklist is signed.**

The point of the dry run is that annotating tens of thousands of posts takes
hours, and discovering a batch size problem or a session timeout at that point
is expensive. Better to find it now on data that does not matter.

## 1. Setup

In [ ]:
!nvidia-smi

# Confirm a GPU is actually attached. Colab will happily give you a CPU
# runtime, and the annotation run would then take about forty times longer.
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn pandas matplotlib tqdm

In [ ]:
# Pull the project code.
REPO = "https://github.com/hsnnaw/dissertation.git"
PROJECT = "/content/project"

import os, sys

# Absolute paths throughout, so re-running this cell after the %cd below
# behaves the same as running it fresh.
if os.path.isdir(PROJECT):
    !cd $PROJECT && git pull --ff-only
else:
    !git clone $REPO $PROJECT

%cd $PROJECT
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("ready")


In [ ]:
# Persist outputs to Drive so a session timeout does not lose hours of work.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
WORK = Path("/content/drive/MyDrive/dissertation")
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / "outputs").mkdir(parents=True, exist_ok=True)
print(WORK)

## 2. Mode

Leave this `True` until the checklist is signed.

In [ ]:
DRY_RUN = True

# Small enough to iterate on, large enough for timings to extrapolate.
N_POSTS = 500 if DRY_RUN else None

STRATEGY = "few_shot"          # best on the benchmark: 0.902 at 1.78s/post
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"DRY_RUN={DRY_RUN}  posts={N_POSTS}  strategy={STRATEGY}")

## 3. Input data

In dry run mode this builds a small corpus of neutral text with the same shape
as the real one: a `text` field, a `post_id`, and a `subreddit`. The content is
irrelevant; what matters is that every stage of the pipeline runs and that the
throughput figures are real.

In [ ]:
import json, random
from pathlib import Path

POSTS = Path("data/interim/posts.jsonl")
POSTS.parent.mkdir(parents=True, exist_ok=True)

if DRY_RUN:
    from datasets import load_dataset

    # AG News: ordinary news text, nothing sensitive. Any neutral corpus of
    # comparable length would do equally well.
    ds = load_dataset("fancyzhx/ag_news", split=f"train[:{N_POSTS}]")
    topics = ["world", "sports", "business", "scitech"]

    with POSTS.open("w") as f:
        for i, row in enumerate(ds):
            f.write(json.dumps({
                "post_id": f"dry{i:05d}",
                "text": row["text"],
                "subreddit": topics[row["label"]],
                "group": "non_mh",
            }) + "\n")
    print(f"dry run corpus: {N_POSTS} posts")

else:
    # Real corpus. Requires the signed checklist.
    # Download the dataset to data/raw/ first, then:
    from src.data import prepare
    prepare(
        input_dir=Path("data/raw"),
        output_path=POSTS,
        sample=20000,
    )

print(sum(1 for _ in POSTS.open()), "posts ready")

## 4. Annotation

Runs the annotator locally on the Colab GPU. No text leaves the machine,
which is the condition the ethics route depends on.

`MODEL_ID` above is set to `Qwen/Qwen2.5-7B-Instruct`, which is ungated. The
dry run measures throughput and pipeline plumbing rather than label quality,
so the model choice does not affect what it tells us.

For the real annotation run, switch `MODEL_ID` to
`meta-llama/Llama-3.1-8B-Instruct`. That one is licence-gated: accept the
licence on the model page first, then supply a read token below.

The token cell is worth running either way. Qwen does not require it, but an
authenticated session gets higher rate limits and faster downloads.


In [ ]:
from huggingface_hub import login
login()  # paste a token with read access

In [ ]:
import time
from src.annotate import run

LABELS = Path(f"data/processed/labels_{STRATEGY}.jsonl")

# Annotation is resumable, so rerunning this cell on a finished file does
# nothing and times nothing. Delete the file to measure a fresh run.
REDO = False
if REDO and LABELS.exists():
    LABELS.unlink()
    print(f"deleted {LABELS}, annotating from scratch")

started = time.perf_counter()
stats = run(
    input_path=POSTS,
    output_path=LABELS,
    strategy=STRATEGY,
    backend_kind="transformers",
    model=MODEL_ID,
    batch_size=16,
)
elapsed = time.perf_counter() - started

n = stats["ok"] + stats["failed"]

if n == 0:
    print("\nNothing to annotate: every post in POSTS is already in LABELS.")
    print("Set REDO = True above to time a fresh run.")
else:
    # Separate the one-off cost from the per-post cost. Downloading and
    # loading the model happens once whether the corpus is 500 posts or
    # 20,000, so extrapolating from wall clock overstates the real run.
    annotation_s = stats["total_latency_s"]
    setup_s = elapsed - annotation_s

    print(f"\n{n} posts in {elapsed/60:.1f} min wall clock")
    print(f"  model download and load : {setup_s/60:5.1f} min  (one-off)")
    print(f"  annotation              : {annotation_s/60:5.1f} min  "
          f"= {annotation_s/n:.3f} s/post")
    print(f"\nextrapolated to 20,000 posts: "
          f"{annotation_s/n*20000/3600:.1f} hours annotation "
          f"+ {setup_s/60:.0f} min setup")
    print(f"parse failure rate: {stats.get('failure_rate', 0):.1%}")

# Token spread reads from the file, so it works whether or not anything was
# annotated just now. Check it before lowering max_tokens: cutting below the
# longest real response truncates valid JSON into a parse failure.
toks = sorted(r["output_tokens"] for r in map(json.loads, LABELS.open())
              if "output_tokens" in r)
if toks:
    p = lambda q: toks[min(int(len(toks) * q), len(toks) - 1)]
    cap = 128  # the few_shot default in src/annotate.py
    print(f"\noutput tokens over {len(toks)} records")
    print(f"  median={p(0.5)}  p90={p(0.9)}  p99={p(0.99)}  max={toks[-1]}")
    print(f"  at or above the {cap}-token cap: {sum(t >= cap for t in toks)}")


**Read the extrapolation before continuing.** If 20,000 posts would take longer
than a Colab session allows, the options are a larger batch size, a smaller
corpus, or splitting the run across sessions. Annotation is resumable, so the
last of those works: rerun the same cell and it picks up where it stopped.

A parse failure rate above a few percent means the prompt needs attention
before committing to the full run.

## 5. Splits

In [ ]:
from src.splits import load_labelled, stratified_split, subreddit_split, summarise

# The annotation output carries labels keyed by post_id but not the post
# text, so join it back from POSTS. Training needs both.
records = load_labelled(LABELS, posts_path=POSTS)
print(f"{len(records)} labelled records")

for mode, splitter in [("random", stratified_split), ("subreddit", subreddit_split)]:
    splits = splitter(records)
    summarise(splits, "is_disclosure")
    outdir = Path(f"data/processed/splits_{mode}")
    outdir.mkdir(parents=True, exist_ok=True)
    for name, group in splits.items():
        with (outdir / f"{name}.jsonl").open("w") as f:
            for r in group:
                f.write(json.dumps(r) + "\n")
    print(f"wrote {outdir}\n")


**Check the class balance before training.** If the positive rate is below
about 5%, the classifier will struggle regardless of class weighting, and the
corpus sampling needs rethinking rather than the model.

In dry run mode expect a very low positive rate, since news text contains no
self-disclosure. That is the correct result: it confirms the annotator is not
firing on everything.

## 6. Training

In [ ]:
from src.train import train

result = train(
    splits_dir=Path("data/processed/splits_random"),
    output_dir=Path("outputs/experiments/gen_seen"),
    model_name="roberta-base",
    epochs=3,
    batch_size=16,
)

## 7. Experiment grid

Skip in dry run mode. The numbers would be meaningless and it costs an hour.

In [ ]:
if not DRY_RUN:
    # Noise robustness
    for rate in [0.0, 0.05, 0.10, 0.20, 0.30]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/noise_{rate}"),
            noise_rate=rate,
        )

    # Generalisation
    train(
        splits_dir=Path("data/processed/splits_subreddit"),
        output_dir=Path("outputs/experiments/gen_unseen"),
    )

    # Size against cost
    for m in ["roberta-base", "distilroberta-base"]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/size_{m}"),
            model_name=m,
        )

    # Class weighting ablation
    train(
        splits_dir=Path("data/processed/splits_random"),
        output_dir=Path("outputs/experiments/no_weights"),
        class_weights=False,
    )
else:
    print("skipped in dry run")

## 8. Analysis

In [ ]:
!python -m scripts.analyse breakdown --dir outputs/experiments/gen_seen --plot

if not DRY_RUN:
    !python -m scripts.analyse noise --dir outputs/experiments
    !python -m scripts.analyse cost --dir outputs/experiments --llm-ms 1780
    !python -m scripts.analyse gap  --dir outputs/experiments
    !python -m scripts.collect_results --dir outputs/experiments

## 9. Save to Drive

In [ ]:
import shutil

# Colab sessions are wiped without warning. Copy anything expensive to Drive
# as soon as it exists, not at the end of the notebook.
#
# What is deliberately NOT copied:
#
#   splits_*      contain verbatim post text, needed for training. They
#                 regenerate from the labels in seconds, so persisting them
#                 buys nothing and puts post text on third-party storage.
#   model,        half a gigabyte per training run, ten runs in the grid,
#   checkpoints   and nothing downstream reads them.
#
# The labels file IS copied and is the one thing that must survive: it costs
# hours to regenerate and carries no post text, only post_id and labels.
SKIP = shutil.ignore_patterns("splits_*", "model", "checkpoints", "checkpoint-*")

for src in ["outputs", "data/processed"]:
    dst = WORK / src.replace("/", "_")
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=SKIP)
        print(f"{src} -> {dst}")

# Fail loudly if post text reached Drive anyway, rather than discovering it
# later. Checks for the text field the training splits carry.
leaked = [f for f in WORK.rglob("*.jsonl")
          if any('"text":' in line for line in f.open().readlines()[:5])]
if leaked:
    print("\nWARNING: files on Drive contain post text:")
    for f in leaked:
        print(f"  {f}")

used = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file())
print(f"\n{used / 1e6:.1f} MB in {WORK}")
